# Beach robot CNN DQN training

This notebook calls the shared training harness used by `robots_Q_learning.py`. It records held-out metrics and exports the best evaluation checkpoint.

In [1]:
from pathlib import Path
import json
import sys
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'DeepQBeachRobots').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'DeepQBeachRobots').is_dir():
    raise RuntimeError(f'Cannot locate project root from {Path.cwd()}')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from DeepQBeachRobots.robots_Q_learning import DqnConfig, train

SEED = 7
CONFIG = DqnConfig()
print(CONFIG)

2026-10-21 09:12:16.738177: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-10-21 09:12:16.738250: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-10-21 09:12:16.740939: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


DqnConfig(learning_rate=0.0001, gamma=0.99, gradient_clipping=10.0, target_update_period=500, batch_size=64, replay_buffer_capacity=100000, initial_collect_steps=5000, epsilon_initial=1.0, epsilon_final=0.05, epsilon_decay_steps=50000)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [2]:
# Run these stages in order. A stage is a separate, reproducible benchmark.
CURRICULUM = {
    'stage_1': {
        'n_obstacles': 0, 'n_small_rubbish': 10, 'width': 10, 'height': 10,
        'max_steps': 200, 'observation_radius': 5,
    },
    'stage_2': {
        'n_obstacles': 2, 'n_small_rubbish': 20, 'width': 10, 'height': 10,
        'max_steps': 300, 'observation_radius': 5,
    },
    'stage_3': {
        'n_obstacles': 8, 'n_small_rubbish': 40, 'width': 20, 'height': 20,
        'max_steps': 500, 'observation_radius': 5,
    },
}
STAGE_NAME = 'stage_1'
ENVIRONMENT = CURRICULUM[STAGE_NAME]
ENVIRONMENT

{'n_obstacles': 0,
 'n_small_rubbish': 10,
 'width': 10,
 'height': 10,
 'max_steps': 200,
 'observation_radius': 5}

In [ ]:
# Use 200_000 updates for a full run. Keep the output directory unique per stage/run.
OUTPUT_DIRECTORY = Path('training_runs') / STAGE_NAME
agent, result = train(
    num_iterations=200_000,
    output_directory=OUTPUT_DIRECTORY,
    seed=SEED,
    environment_kwargs=ENVIRONMENT,
    config=CONFIG,
    evaluation_interval=10_000,
)
result['best_checkpoint_path']

/usr/local/lib/python3.11/site-packages/tf_agents/specs/array_spec.py:352: RuntimeWarning: invalid value encountered in cast
  self._minimum[self._minimum == -np.inf] = low
/usr/local/lib/python3.11/site-packages/tf_agents/specs/array_spec.py:353: RuntimeWarning: invalid value encountered in cast
  self._minimum[self._minimum == np.inf] = high
/usr/local/lib/python3.11/site-packages/tf_agents/specs/array_spec.py:355: RuntimeWarning: invalid value encountered in cast
  self._maximum[self._maximum == -np.inf] = low
/usr/local/lib/python3.11/site-packages/tf_agents/specs/array_spec.py:356: RuntimeWarning: invalid value encountered in cast
  self._maximum[self._maximum == np.inf] = high


Instructions for updating:
Use `as_dataset(..., single_deterministic_pass=False) instead.
Instructions for updating:
back_prop=False is deprecated. Consider using tf.stop_gradient instead.
Instead of:
results = tf.foldr(fn, elems, back_prop=False)
Use:
results = tf.nest.map_structure(tf.stop_gradient, tf.foldr(fn, elems))
step=1000 loss_100=0.0352
step=2000 loss_100=0.3284
step=3000 loss_100=0.5598
step=4000 loss_100=0.3546
step=5000 loss_100=0.2635
step=6000 loss_100=0.3354
step=7000 loss_100=0.3956
step=8000 loss_100=0.3970
step=9000 loss_100=0.4672
step=10000 loss_100=0.4431
evaluation={'average_reward': 100.96666666666646, 'average_picked': 8.966666666666667, 'average_failed_moves': 11.433333333333334, 'average_steps': 116.23333333333333, 'completion_rate': 0.5333333333333333, 'episodes': 30, 'iteration': 10000}
step=11000 loss_100=0.5147
step=12000 loss_100=0.5075
step=13000 loss_100=0.5457


In [ ]:
metrics = json.loads((OUTPUT_DIRECTORY / 'training_metrics.json').read_text())
print('Baselines:', metrics['baselines'])
print('Best checkpoint:', metrics['best_checkpoint_path'])
metrics['evaluations']

In [ ]:
evaluations = metrics['evaluations']
iterations = [entry['iteration'] for entry in evaluations]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot(iterations, [entry['average_picked'] for entry in evaluations], marker='o')
axes[0].set(title='Held-out rubbish picked', xlabel='Training step', ylabel='Average picked')
axes[1].plot(iterations, [entry['completion_rate'] for entry in evaluations], marker='o')
axes[1].set(title='Held-out completion rate', xlabel='Training step', ylabel='Rate')
axes[2].plot(iterations, [entry['average_failed_moves'] for entry in evaluations], marker='o')
axes[2].set(title='Held-out failed moves', xlabel='Training step', ylabel='Average failed moves')
plt.tight_layout()